In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [1]:
import logging
import os
import re
import sys
import numpy as np
from itertools import chain
from gensim.models import KeyedVectors
import gensim
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import pickle

# =================== 超参（和你本地保持一致） ===================
embed_size = 300
max_len = 512

# =================== Kaggle路径【自行核对修改】 ===================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
GLOVE_PATH = "/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt"

# =================== 文本清洗函数（原版不动） ===================
def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words

def encode_samples(tokenized_samples, word_to_idx):
    features = []
    for sample in tokenized_samples:
        feature = []
        for token in sample:
            if token in word_to_idx:
                feature.append(word_to_idx[token])
            else:
                feature.append(0)
        features.append(feature)
    return features

def pad_samples(features, maxlen=max_len, PAD=0):
    padded_features = []
    for feature in features:
        if len(feature) >= maxlen:
            padded_feature = feature[:maxlen]
        else:
            padded_feature = feature.copy()
            while len(padded_feature) < maxlen:
                padded_feature.append(PAD)
        padded_features.append(padded_feature)
    return padded_features

# =================== 主流程 ===================
os.makedirs("/kaggle/working/pickle", exist_ok=True)

train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

clean_train_reviews, train_labels = [], []
for i, review in enumerate(train["review"]):
    clean_train_reviews.append(review_to_wordlist(review))
    train_labels.append(train["sentiment"][i])

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review))

vocab = set(chain(*clean_train_reviews)) | set(chain(*clean_test_reviews))
vocab_size = len(vocab)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    clean_train_reviews, train_labels, test_size=0.2, random_state=0)

# ===================【重点修改】适配Common Crawl 840B（glove-gensim分割逻辑） ===================
wvmodel = KeyedVectors(embed_size)
word_dict = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = line.split()
        if len(tokens) <= embed_size:
            continue
        try:
            vec = np.array(tokens[-embed_size:], dtype=np.float32)
            word = " ".join(tokens[:-embed_size])
            word_dict[word] = vec
        except ValueError:
            continue
wvmodel.add_vectors(list(word_dict.keys()), list(word_dict.values()))
print(f"GloVe加载完成，载入词总数：{len(wvmodel)}")
# =========================================================================================

word_to_idx = {word: i + 1 for i, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0
idx_to_word = {i + 1: word for i, word in enumerate(vocab)}
idx_to_word[0] = '<unk>'

train_features = torch.tensor(pad_samples(encode_samples(train_reviews, word_to_idx)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews, word_to_idx)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test_reviews, word_to_idx)))

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 构建Embedding权重矩阵
weight = torch.zeros(vocab_size + 1, embed_size)
hit = 0
for word, idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))
        hit += 1
print(f"词表匹配成功向量：{hit}/{len(word_to_idx)}")

pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
pickle.dump(
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab],
    open(pickle_file, 'wb'))
print('pickle文件生成完成！')

GloVe加载完成，载入词总数：2195895


/tmp/ipykernel_58/1117333043.py:114: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))


词表匹配成功向量：77554/101400
pickle文件生成完成！


In [3]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from tqdm import tqdm

from sklearn.metrics import accuracy_score

# =================== 超参 ===================
num_epochs = 10
embed_size = 300
num_hiddens = 128
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.01

# =================== Kaggle 路径 ===================
PICKLE_PATH = "/kaggle/working/pickle/imdb_glove.pickle3"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/attention_lstm.csv"

# =================== 创建目录 ===================
os.makedirs("/kaggle/working/result", exist_ok=True)

# =================== 设备 ===================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_gpu = torch.cuda.is_available()

print(f"Using device: {device}")

# =================== 模型定义 ===================
class Attention(nn.Module):
    def __init__(self, num_hiddens, bidirectional, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.bidirectional = bidirectional

        if self.bidirectional:
            self.w_omega = nn.Parameter(torch.Tensor(num_hiddens * 2, num_hiddens * 2))
            self.u_omega = nn.Parameter(torch.Tensor(num_hiddens * 2, 1))
        else:
            self.w_omega = nn.Parameter(torch.Tensor(num_hiddens, num_hiddens))
            self.u_omega = nn.Parameter(torch.Tensor(num_hiddens, 1))

        nn.init.uniform_(self.w_omega, -0.1, 0.1)
        nn.init.uniform_(self.u_omega, -0.1, 0.1)

    def forward(self, inputs):
        x = inputs
        u = torch.tanh(torch.matmul(x, self.w_omega))
        att = torch.matmul(u, self.u_omega)
        att_score = F.softmax(att, dim=1)
        outputs = x * att_score
        return outputs

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.embed_size = embed_size
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False

        self.encoder = nn.LSTM(
            input_size=self.embed_size,
            hidden_size=self.num_hiddens,
            num_layers=self.num_layers,
            bidirectional=self.bidirectional,
            dropout=0
        )

        self.attention = Attention(
            num_hiddens=self.num_hiddens,
            bidirectional=self.bidirectional
        )

        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 4, labels)
        else:
            self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)

        states, _ = self.encoder(embeddings.permute(1, 0, 2))

        attention = self.attention(states)

        encoding = torch.cat([attention[0], attention[-1]], dim=1)

        outputs = self.decoder(encoding)

        return outputs

# =================== 主训练流程 ===================
if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(
        format='%(asctime)s: %(levelname)s: %(message)s',
        level=logging.INFO
    )

    logger.info(r"running %s" % ''.join(sys.argv))

    # =================== 读取 pickle ===================
    logger.info('loading pickle data...')

    [
        train_features, train_labels,
        val_features, val_labels,
        test_features, weight,
        word_to_idx, idx_to_word, vocab
    ] = pickle.load(open(PICKLE_PATH, 'rb'))

    logger.info('pickle data loaded!')

    # =================== 读取 test id ===================
    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

    # =================== 模型初始化 ===================
    net = SentimentNet(
        embed_size=embed_size,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        labels=labels
    )

    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)

    # =================== DataLoader ===================
    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    # =================== 训练循环 ===================
    for epoch in range(num_epochs):
        start = time.time()

        train_loss, val_losses = 0.0, 0.0
        train_acc, val_acc = 0.0, 0.0
        n, m = 0, 0

        net.train()

        with tqdm(total=len(train_iter), desc=f'Epoch {epoch}') as pbar:
            for feature, label in train_iter:
                n += 1

                feature = feature.to(device)
                label = label.to(device)

                score = net(feature)
                loss = loss_function(score, label)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                train_acc += accuracy_score(
                    torch.argmax(score.cpu().data, dim=1),
                    label.cpu()
                )

                train_loss += loss.item()

                pbar.set_postfix({
                    'epoch': '%d' % epoch,
                    'train loss': '%.4f' % (train_loss / n),
                    'train acc': '%.4f' % (train_acc / n)
                })

                pbar.update(1)

        # =================== 验证 ===================
        net.eval()

        with torch.no_grad():
            for val_feature, val_label in val_iter:
                m += 1

                val_feature = val_feature.to(device)
                val_label = val_label.to(device)

                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)

                val_acc += accuracy_score(
                    torch.argmax(val_score.cpu().data, dim=1),
                    val_label.cpu()
                )

                val_losses += val_loss.item()

        end = time.time()
        runtime = end - start

        logger.info(
            "Epoch %d: train_loss %.4f, train_acc %.4f, val_loss %.4f, val_acc %.4f, time %.2f"
            % (
                epoch,
                train_loss / n,
                train_acc / n,
                val_losses / m,
                val_acc / m,
                runtime
            )
        )

    # =================== 测试预测 ===================
    logger.info('start prediction...')

    test_pred = []

    net.eval()

    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)

                test_pred.extend(
                    torch.argmax(test_score.cpu().data, dim=1).numpy().tolist()
                )

                pbar.update(1)

    # =================== 保存结果 ===================
    result_output = pd.DataFrame({
        "id": test["id"],
        "sentiment": test_pred
    })

    result_output.to_csv(RESULT_PATH, index=False, quoting=3)

    logger.info(f'result saved to {RESULT_PATH}')


2026-08-15 01:17:00,224: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-573126b5-a05a-43ff-ab6b-581a7b0ea218.json
2026-08-15 01:17:00,225: INFO: loading pickle data...


Using device: cuda


2026-08-15 01:17:00,790: INFO: pickle data loaded!
Epoch 0: 100%|██████████| 313/313 [00:24<00:00, 13.03it/s, epoch=0, train loss=0.6422, train acc=0.6119]
2026-08-15 01:17:34,260: INFO: Epoch 0: train_loss 0.6422, train_acc 0.6119, val_loss 0.5486, val_acc 0.7455, time 26.02
Epoch 1: 100%|██████████| 313/313 [00:24<00:00, 12.96it/s, epoch=1, train loss=0.4539, train acc=0.8009]
2026-08-15 01:18:00,429: INFO: Epoch 1: train_loss 0.4539, train_acc 0.8009, val_loss 0.3750, val_acc 0.8410, time 26.17
Epoch 2: 100%|██████████| 313/313 [00:25<00:00, 12.42it/s, epoch=2, train loss=0.3716, train acc=0.8478]
2026-08-15 01:18:27,711: INFO: Epoch 2: train_loss 0.3716, train_acc 0.8478, val_loss 0.3786, val_acc 0.8445, time 27.28
Epoch 3: 100%|██████████| 313/313 [00:25<00:00, 12.35it/s, epoch=3, train loss=0.3244, train acc=0.8718]
2026-08-15 01:18:55,139: INFO: Epoch 3: train_loss 0.3244, train_acc 0.8718, val_loss 0.3721, val_acc 0.8659, time 27.43
Epoch 4: 100%|██████████| 313/313 [00:25<00:0

In [4]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# =================== 超参【全部优化】 ===================
num_epochs = 8
embed_size = 300
num_hiddens = 256        # 隐藏层扩大
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.001               # 降低初始学习率
weight_decay = 1e-4      # L2正则
grad_clip = 5.0          # 梯度裁剪
dropout_rate = 0.3

# =================== Kaggle 路径 ===================
PICKLE_PATH = "/kaggle/working/pickle/imdb_glove.pickle3"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/attention_lstm(1).csv"

# =================== 创建目录 ===================
os.makedirs("/kaggle/working/result", exist_ok=True)

# =================== 设备 ===================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_gpu = torch.cuda.is_available()
print(f"Using device: {device}")

# =================== 改进Attention模块 ===================
class Attention(nn.Module):
    def __init__(self, hidden_dim, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output):
        # lstm_output: [seq_len, batch, hidden]
        attn_weights = torch.tanh(self.attn(lstm_output))
        attn_scores = self.v(attn_weights).squeeze(-1)  # [seq_len, batch]
        attn_dist = F.softmax(attn_scores, dim=0)
        weighted = lstm_output * attn_dist.unsqueeze(-1)
        output = torch.sum(weighted, dim=0)  # [batch, hidden]
        return output, attn_dist

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout=0.3, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.embed_size = embed_size
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.dropout = nn.Dropout(dropout)

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = True   # ✅ 允许微调词向量！关键提分点

        self.encoder = nn.LSTM(
            input_size=self.embed_size,
            hidden_size=self.num_hiddens,
            num_layers=self.num_layers,
            bidirectional=self.bidirectional,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=False
        )

        lstm_out_dim = num_hiddens * 2 if bidirectional else num_hiddens
        self.attention = Attention(lstm_out_dim)

        self.decoder = nn.Sequential(
            nn.Linear(lstm_out_dim, lstm_out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_out_dim, labels)
        )

    def forward(self, inputs):
        # inputs: [batch, seq_len]
        embeddings = self.embedding(inputs)  # [batch, seq, embed]
        embeddings = self.dropout(embeddings)
        embeddings = embeddings.permute(1, 0, 2)  # [seq, batch, embed]

        lstm_out, _ = self.encoder(embeddings) # [seq, batch, hidden*2]
        attn_pool, _ = self.attention(lstm_out) # [batch, hidden*2]

        outputs = self.decoder(attn_pool)
        return outputs

# =================== 主训练流程 ===================
if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(
        format='%(asctime)s: %(levelname)s: %(message)s',
        level=logging.INFO
    )
    logger.info(r"running %s" % ''.join(sys.argv))

    logger.info('loading pickle data...')
    [
        train_features, train_labels,
        val_features, val_labels,
        test_features, weight,
        word_to_idx, idx_to_word, vocab
    ] = pickle.load(open(PICKLE_PATH, 'rb'))
    logger.info('pickle data loaded!')

    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

    # =================== 模型初始化 ===================
    net = SentimentNet(
        embed_size=embed_size,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        labels=labels,
        dropout=dropout_rate
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    # 学习率衰减
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    # =================== DataLoader ===================
    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    # =================== 训练循环 ===================
    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0.0, 0.0
        train_acc, val_acc = 0.0, 0.0
        n, m = 0, 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch}') as pbar:
            for feature, label in train_iter:
                n += 1
                feature = feature.to(device)
                label = label.to(device)

                score = net(feature)
                loss = loss_function(score, label)

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), max_norm=grad_clip) # ✅梯度裁剪
                optimizer.step()

                train_acc += accuracy_score(
                    torch.argmax(score.cpu().data, dim=1),
                    label.cpu()
                )
                train_loss += loss.item()

                pbar.set_postfix({
                    'train loss': '%.4f' % (train_loss / n),
                    'train acc': '%.4f' % (train_acc / n)
                })
                pbar.update(1)

        # =================== 验证 ===================
        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                m += 1
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_acc += accuracy_score(
                    torch.argmax(val_score.cpu().data, dim=1),
                    val_label.cpu()
                )
                val_losses += val_loss.item()

        scheduler.step() # 更新学习率
        end = time.time()
        runtime = end - start

        logger.info(
            "Epoch %d: train_loss %.4f, train_acc %.4f, val_loss %.4f, val_acc %.4f, time %.2f, lr=%.6f"
            % (
                epoch,
                train_loss / n,
                train_acc / n,
                val_losses / m,
                val_acc / m,
                runtime,
                optimizer.param_groups[0]['lr']
            )
        )

    # =================== 测试预测 ===================
    logger.info('start prediction...')
    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    # =================== 保存结果 ===================
    result_output = pd.DataFrame({
        "id": test["id"],
        "sentiment": test_pred
    })
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)
    logger.info(f'result saved to {RESULT_PATH}')

2026-08-15 01:24:21,508: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-573126b5-a05a-43ff-ab6b-581a7b0ea218.json
2026-08-15 01:24:21,510: INFO: loading pickle data...


Using device: cuda


2026-08-15 01:24:22,083: INFO: pickle data loaded!
Epoch 0: 100%|██████████| 313/313 [01:18<00:00,  3.98it/s, train loss=0.4090, train acc=0.7973]
2026-08-15 01:25:47,968: INFO: Epoch 0: train_loss 0.4090, train_acc 0.7973, val_loss 0.2526, val_acc 0.9009, time 85.22, lr=0.000962
Epoch 1: 100%|██████████| 313/313 [01:21<00:00,  3.83it/s, train loss=0.2454, train acc=0.9019]
2026-08-15 01:27:16,431: INFO: Epoch 1: train_loss 0.2454, train_acc 0.9019, val_loss 0.2494, val_acc 0.9001, time 88.46, lr=0.000854
Epoch 2: 100%|██████████| 313/313 [01:23<00:00,  3.75it/s, train loss=0.1894, train acc=0.9290]
2026-08-15 01:28:46,594: INFO: Epoch 2: train_loss 0.1894, train_acc 0.9290, val_loss 0.2405, val_acc 0.9039, time 90.16, lr=0.000691
Epoch 3: 100%|██████████| 313/313 [01:24<00:00,  3.72it/s, train loss=0.1335, train acc=0.9520]
2026-08-15 01:30:17,411: INFO: Epoch 3: train_loss 0.1335, train_acc 0.9520, val_loss 0.3209, val_acc 0.8879, time 90.82, lr=0.000500
Epoch 4: 100%|██████████| 313

In [4]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# =================== 超参 ===================
num_epochs = 12
embed_size = 300
num_hiddens = 256
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
weight_decay = 1e-4
grad_clip = 5.0
dropout_rate = 0.4
patience = 3
warmup_epochs = 1

# =================== 路径 ===================
PICKLE_PATH = "/kaggle/working/pickle/imdb_glove.pickle3"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
RESULT_PATH = "/kaggle/working/result/attention_lstm_best.csv"
BEST_MODEL_PATH = "/kaggle/working/result/best_model.pth"

os.makedirs("/kaggle/working/result", exist_ok=True)

# =================== 设备 ===================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_gpu = torch.cuda.is_available()

print(f"Using device: {device}")

# =================== Attention ===================
class Attention(nn.Module):
    def __init__(self, hidden_dim, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output, mask=None):
        seq_len, batch, _ = lstm_output.shape
        attn_weights = torch.tanh(self.attn(lstm_output))
        attn_scores = self.v(attn_weights).squeeze(-1)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask.T, -1e4)

        attn_dist = F.softmax(attn_scores, dim=0)
        weighted = lstm_output * attn_dist.unsqueeze(-1)
        output = torch.sum(weighted, dim=0)

        return output, attn_dist

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout=0.3, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.embed_size = embed_size
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.dropout = nn.Dropout(dropout)

        self.embedding = nn.Embedding.from_pretrained(weight, padding_idx=0)
        self.embedding.weight.requires_grad = True

        self.encoder = nn.LSTM(
            input_size=self.embed_size,
            hidden_size=self.num_hiddens,
            num_layers=self.num_layers,
            bidirectional=self.bidirectional,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=False
        )

        lstm_out_dim = num_hiddens * 2 if bidirectional else num_hiddens
        self.attention = Attention(lstm_out_dim)

        self.decoder = nn.Sequential(
            nn.BatchNorm1d(lstm_out_dim),
            nn.Linear(lstm_out_dim, lstm_out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_out_dim, labels)
        )

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = self.dropout(embeddings)
        embeddings = embeddings.permute(1, 0, 2)

        pad_mask = (inputs == 0)

        lstm_out, _ = self.encoder(embeddings)
        attn_pool, _ = self.attention(lstm_out, pad_mask)

        outputs = self.decoder(attn_pool)

        return outputs

# =================== 学习率调度器 ===================
def get_warmup_scheduler(optimizer, warmup_epoch, total_epoch):
    def lr_lambda(epoch):
        if epoch < warmup_epoch:
            return (epoch + 1) / warmup_epoch
        else:
            progress = (epoch - warmup_epoch) / (total_epoch - warmup_epoch)
            return 0.5 * (1 + torch.cos(torch.tensor(progress * torch.pi)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# =================== 主训练 ===================
if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(
        format='%(asctime)s: %(levelname)s: %(message)s',
        level=logging.INFO
    )
    logger.info(r"running %s" % ''.join(sys.argv))

    logger.info('loading pickle data...')
    [
        train_features, train_labels,
        val_features, val_labels,
        test_features, weight,
        word_to_idx, idx_to_word, vocab
    ] = pickle.load(open(PICKLE_PATH, 'rb'))
    logger.info('pickle data loaded!')

    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

    net = SentimentNet(
        embed_size=embed_size,
        num_hiddens=num_hiddens,
        num_layers=num_layers,
        bidirectional=bidirectional,
        weight=weight,
        labels=labels,
        dropout=dropout_rate
    )
    net.to(device)

    loss_function = nn.CrossEntropyLoss(label_smoothing=0.05)

    optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_warmup_scheduler(optimizer, warmup_epochs, num_epochs)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

    best_val_acc = 0.0
    trigger_times = 0

    for epoch in range(num_epochs):
        start = time.time()

        train_loss = 0.0
        train_acc = 0.0
        n = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch}') as pbar:
            for feature, label in train_iter:
                n += 1
                feature = feature.to(device)
                label = label.to(device)

                score = net(feature)
                loss = loss_function(score, label)

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), max_norm=grad_clip)
                optimizer.step()

                train_acc += accuracy_score(
                    torch.argmax(score.cpu().data, dim=1),
                    label.cpu()
                )
                train_loss += loss.item()

                pbar.set_postfix({
                    'train loss': '%.4f' % (train_loss / n),
                    'train acc': '%.4f' % (train_acc / n)
                })
                pbar.update(1)

        val_losses = 0.0
        val_acc = 0.0
        m = 0

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                m += 1
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)

                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)

                val_acc += accuracy_score(
                    torch.argmax(val_score.cpu().data, dim=1),
                    val_label.cpu()
                )
                val_losses += val_loss.item()

        scheduler.step()

        end = time.time()
        runtime = end - start

        epoch_train_acc = train_acc / n
        epoch_val_acc = val_acc / m

        logger.info(
            "Epoch %d: train_loss %.4f, train_acc %.4f, val_loss %.4f, val_acc %.4f, time %.2f, lr=%.6f"
            % (
                epoch,
                train_loss / n,
                epoch_train_acc,
                val_losses / m,
                epoch_val_acc,
                runtime,
                optimizer.param_groups[0]['lr']
            )
        )

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            trigger_times = 0
            torch.save(net.state_dict(), BEST_MODEL_PATH)
            logger.info(f"Save best model! Best val acc: {best_val_acc:.4f}")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                logger.info(f"Early stop at epoch {epoch}!")
                break

    logger.info('load best model for prediction...')
    net.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

    logger.info('start prediction...')
    test_pred = []

    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    result_output = pd.DataFrame({
        "id": test["id"],
        "sentiment": test_pred
    })
    result_output.to_csv(RESULT_PATH, index=False, quoting=3)

    logger.info(f'result saved to {RESULT_PATH}')
    logger.info(f"Best Validation Accuracy: {best_val_acc:.4f}")


Using device: cuda


Prediction: 100%|██████████| 391/391 [00:34<00:00, 11.48it/s]
